# Bayan Student Notebook — 03_text_classification.ipynb

**Student:** Anas Ibrahim Al-Mutairi  
**GitHub:** `Ans212121`  
**Source:** reorganised from my own executed combined Bayan Colab notebook.

This corrected version keeps my topic-classification work and adds a separate sentiment head required by the Bayan specification. Run top-to-bottom in Colab and save the outputs.


# Day 2 — Text Classification

This section starts Day 2 and focuses on text classification using a baseline model and multilingual DistilBERT.

Notebook 1 and Notebook 2 from Day 1 were completed successfully.

In [25]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة

import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}

needs_install = []

for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None

    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *needs_install
        ]
    )

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is required. Open this notebook in Google Colab."
    )

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


In [26]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [27]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day2_classification.csv"

FALLBACK_ROWS = [
    {
        "example_id": "F-001",
        "group_id": "DG-A",
        "split": "train",
        "language": "ar",
        "text": "تعذر تسجيل الدخول إلى البوابة",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-002",
        "group_id": "DG-A",
        "split": "train",
        "language": "en",
        "text": "I cannot sign in to the portal",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-003",
        "group_id": "DG-B",
        "split": "train",
        "language": "ar",
        "text": "الخدمة الإلكترونية سريعة وواضحة",
        "topic": "digital_service",
        "sentiment": "positive",
    },
    {
        "example_id": "F-004",
        "group_id": "DG-B",
        "split": "train",
        "language": "en",
        "text": "The online service is clear and fast",
        "topic": "digital_service",
        "sentiment": "positive",
    },
    {
        "example_id": "F-005",
        "group_id": "PG-A",
        "split": "train",
        "language": "ar",
        "text": "أحتاج معرفة حالة طلب التصريح",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-006",
        "group_id": "PG-A",
        "split": "train",
        "language": "en",
        "text": "I need the status of my permit request",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-007",
        "group_id": "PG-B",
        "split": "train",
        "language": "ar",
        "text": "تمت الموافقة على التصريح اليوم",
        "topic": "permit",
        "sentiment": "positive",
    },
    {
        "example_id": "F-008",
        "group_id": "PG-B",
        "split": "train",
        "language": "en",
        "text": "The permit was approved today",
        "topic": "permit",
        "sentiment": "positive",
    },
    {
        "example_id": "F-009",
        "group_id": "HG-A",
        "split": "train",
        "language": "ar",
        "text": "تأخر موعد العيادة هذا الصباح",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-010",
        "group_id": "HG-A",
        "split": "train",
        "language": "en",
        "text": "My clinic appointment was delayed",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-011",
        "group_id": "HG-B",
        "split": "train",
        "language": "ar",
        "text": "كانت خدمة العيادة ممتازة",
        "topic": "health",
        "sentiment": "positive",
    },
    {
        "example_id": "F-012",
        "group_id": "HG-B",
        "split": "train",
        "language": "en",
        "text": "The clinic service was excellent",
        "topic": "health",
        "sentiment": "positive",
    },
    {
        "example_id": "F-013",
        "group_id": "TG-A",
        "split": "train",
        "language": "ar",
        "text": "الحافلة لم تصل في الوقت المحدد",
        "topic": "transport",
        "sentiment": "negative",
    },
    {
        "example_id": "F-014",
        "group_id": "TG-A",
        "split": "train",
        "language": "en",
        "text": "The bus did not arrive on time",
        "topic": "transport",
        "sentiment": "negative",
    },
    {
        "example_id": "F-015",
        "group_id": "TG-B",
        "split": "train",
        "language": "ar",
        "text": "كانت الرحلة مريحة ومنظمة",
        "topic": "transport",
        "sentiment": "positive",
    },
    {
        "example_id": "F-016",
        "group_id": "TG-B",
        "split": "train",
        "language": "en",
        "text": "The trip was comfortable and organised",
        "topic": "transport",
        "sentiment": "positive",
    },
    {
        "example_id": "F-017",
        "group_id": "DG-V",
        "split": "validation",
        "language": "ar",
        "text": "لم يصل رمز التحقق الرقمي",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-018",
        "group_id": "PG-V",
        "split": "validation",
        "language": "en",
        "text": "How can I renew the permit",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-019",
        "group_id": "HG-V",
        "split": "validation",
        "language": "ar",
        "text": "أحتاج إعادة جدولة الموعد الصحي",
        "topic": "health",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-020",
        "group_id": "TG-V",
        "split": "validation",
        "language": "en",
        "text": "The bus route has changed",
        "topic": "transport",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-021",
        "group_id": "DG-T",
        "split": "test",
        "language": "en",
        "text": "The verification code did not arrive",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-022",
        "group_id": "PG-T",
        "split": "test",
        "language": "ar",
        "text": "تأخر إصدار التصريح المطلوب",
        "topic": "permit",
        "sentiment": "negative",
    },
    {
        "example_id": "F-023",
        "group_id": "HG-T",
        "split": "test",
        "language": "en",
        "text": "The clinic appointment was cancelled",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-024",
        "group_id": "TG-T",
        "split": "test",
        "language": "ar",
        "text": "توقفت الحافلة قبل المحطة",
        "topic": "transport",
        "sentiment": "negative",
    },
]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
        rows = list(csv.DictReader(io.StringIO(text)))

    DATA_SOURCE = "github_course_file"

except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))

assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

Data source: github_course_file
Rows: 40
Topics: Counter({'digital_service': 10, 'permit': 10, 'health': 10, 'transport': 10})


In [28]:
def validate_splits(rows):
    required = {"train", "validation", "test"}

    group_owner = {}
    labels_by_split = {
        name: set()
        for name in required
    }

    counts = Counter()

    for row in rows:
        split = row["split"]
        group = row["group_id"]
        label = row["topic"]

        if split not in required:
            raise ValueError(
                f"Unknown split: {split}"
            )

        previous = group_owner.setdefault(
            group,
            split
        )

        if previous != split:
            raise ValueError(
                f"Group leakage: {group}"
            )

        labels_by_split[split].add(label)
        counts[split] += 1

    all_labels = set().union(
        *labels_by_split.values()
    )

    for split in required:
        if labels_by_split[split] != all_labels:
            raise ValueError(
                f"Missing label in {split}"
            )

    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
    }


split_report = validate_splits(rows)

print(
    json.dumps(
        split_report,
        ensure_ascii=False,
        indent=2
    )
)

assert split_report["group_overlap"] == 0

print("Split contract=PASS")

{
  "rows": {
    "train": 24,
    "validation": 8,
    "test": 8
  },
  "groups": 20,
  "labels": [
    "digital_service",
    "health",
    "permit",
    "transport"
  ],
  "group_overlap": 0
}
Split contract=PASS


In [29]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["topic"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["topic"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

Baseline validation macro-F1 (MEASURED_SMOKE): 0.6667
Baseline=PASS


In [30]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
NUM_EPOCHS = 2 if DEVICE.type == "cuda" else 12

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "partial_finetune_cpu"
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print("Epochs:", NUM_EPOCHS)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Training mode: partial_finetune_cpu
Epochs: 12
Trainable parameters: 7,681,540 / 135,327,748


In [31]:
def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["topic"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

Batch functions=PASS


In [32]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 1e-4
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_parameters, lr=learning_rate)
losses = []
train_steps = 0
epoch_history = []
best_validation_f1 = -1.0
best_epoch = 0
best_trainable_state = None

for epoch_index in range(NUM_EPOCHS):
    model.train()
    epoch_losses = []
    for batch in iter_batches(
        train_rows, BATCH_SIZE, shuffle=True, seed=SEED + epoch_index + 1
    ):
        optimizer.zero_grad(set_to_none=True)
        encoded = encode_batch(batch)
        output = model(**encoded)
        loss = output.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        epoch_losses.append(loss_value)
        train_steps += 1

    epoch_predictions = predict(validation_rows)
    epoch_f1 = f1_score(
        [row["topic"] for row in validation_rows], epoch_predictions,
        labels=LABELS, average="macro", zero_division=0,
    )
    epoch_history.append({
        "epoch": epoch_index + 1,
        "mean_loss": float(np.mean(epoch_losses)),
        "validation_macro_f1": float(epoch_f1),
    })
    if epoch_f1 > best_validation_f1:
        best_validation_f1 = float(epoch_f1)
        best_epoch = epoch_index + 1
        if TRAINING_MODE == "partial_finetune_cpu":
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad
            }
    print(
        f"epoch={epoch_index + 1:02d} mean_loss={np.mean(epoch_losses):.4f} "
        f"validation_macro_f1={epoch_f1:.4f}"
    )

if best_trainable_state is not None:
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_trainable_state:
                parameter.copy_(best_trainable_state[name].to(parameter.device))
    selected_epoch = best_epoch
else:
    selected_epoch = NUM_EPOCHS

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
assert any(parameter.requires_grad for parameter in model.base_model.parameters())
print("Transformer optimizer steps=PASS", {"steps": train_steps, "selected_epoch": selected_epoch})

epoch=01 mean_loss=1.4031 validation_macro_f1=0.2917
epoch=02 mean_loss=1.3156 validation_macro_f1=0.4167
epoch=03 mean_loss=1.2396 validation_macro_f1=0.4167
epoch=04 mean_loss=1.1272 validation_macro_f1=0.6250
epoch=05 mean_loss=0.9523 validation_macro_f1=0.6250
epoch=06 mean_loss=0.7202 validation_macro_f1=0.7500
epoch=07 mean_loss=0.4770 validation_macro_f1=0.8667
epoch=08 mean_loss=0.2488 validation_macro_f1=0.8667
epoch=09 mean_loss=0.1160 validation_macro_f1=1.0000
epoch=10 mean_loss=0.0552 validation_macro_f1=1.0000
epoch=11 mean_loss=0.0225 validation_macro_f1=1.0000
epoch=12 mean_loss=0.0115 validation_macro_f1=0.8667
Transformer optimizer steps=PASS {'steps': 72, 'selected_epoch': 9}


In [33]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
validation_delta = transformer_val_f1 - baseline_val_f1
print("Selected epoch:", selected_epoch)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation delta vs TF-IDF baseline:", round(validation_delta, 4))
print("Validation predictions:", list(zip(
    [row["topic"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

Selected epoch: 9
Transformer validation macro-F1 (MEASURED_SMOKE): 1.0
Validation delta vs TF-IDF baseline: 0.3333
Validation predictions: [('digital_service', 'digital_service'), ('digital_service', 'digital_service'), ('permit', 'permit'), ('permit', 'permit'), ('health', 'health'), ('health', 'health'), ('transport', 'transport'), ('transport', 'transport')]


In [34]:
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_beaten_on_validation": bool(validation_delta > 0),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "small validation set used for epoch selection",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path("day2_classification_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "result_type": "MEASURED_SMOKE",
  "data_source": "github_course_file",
  "model_id": "distilbert/distilbert-base-multilingual-cased",
  "device": "cpu",
  "training_mode": "partial_finetune_cpu",
  "seed": 42,
  "epochs_run": 12,
  "selected_epoch": 9,
  "train_steps": 72,
  "mean_train_loss": 0.6407361574496867,
  "baseline_validation_macro_f1": 0.6666666666666666,
  "transformer_validation_macro_f1": 1.0,
  "validation_delta_vs_baseline": 0.33333333333333337,
  "baseline_beaten_on_validation": true,
  "baseline_test_macro_f1": 0.7333333333333333,
  "transformer_test_macro_f1": 0.8666666666666667,
  "transformer_test_accuracy": 0.875,
  "limitations": [
    "synthetic tiny dataset",
    "small validation set used for epoch selection",
    "not an estimate of production quality"
  ]
}


800

In [ ]:
# Save traceable topic evidence and the actual project classifier for Notebook 08.
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_test_macro_f1": float(f1_score(test_truth, baseline_test_pred, labels=LABELS, average="macro", zero_division=0)),
    "transformer_test_macro_f1": float(f1_score(test_truth, transformer_test_pred, labels=LABELS, average="macro", zero_division=0)),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
}
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
(reports_dir / "observed_classification.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(results, ensure_ascii=False, indent=2))

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    PROJECT_ARTIFACT_ROOT = Path("/content/drive/MyDrive/Bayan_project_artifact")
except Exception:
    PROJECT_ARTIFACT_ROOT = Path("/content/bayan_project_artifact")

TOPIC_MODEL_DIR = PROJECT_ARTIFACT_ROOT / "topic_model"
TOPIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(TOPIC_MODEL_DIR)
tokenizer.save_pretrained(TOPIC_MODEL_DIR)

PROJECT_VALIDATION_CSV = PROJECT_ARTIFACT_ROOT / "topic_validation.csv"
with PROJECT_VALIDATION_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["example_id", "split", "language", "text", "label"])
    writer.writeheader()
    for row in validation_rows:
        writer.writerow({
            "example_id": row["example_id"],
            "split": "validation",
            "language": row["language"],
            "text": row["text"],
            "label": row["topic"],
        })
print("TOPIC_PROJECT_ARTIFACT_SAVED", TOPIC_MODEL_DIR)
print("PROJECT_VALIDATION_CSV_SAVED", PROJECT_VALIDATION_CSV)


In [35]:
core_checks = {
    "split_isolation": split_report["group_overlap"] == 0,
    "baseline_valid": 0.0 <= baseline_val_f1 <= 1.0,
    "training_ran": train_steps >= 1,
    "transformer_weights_updated": any(
        parameter.requires_grad for parameter in model.base_model.parameters()
    ),
    "loss_is_finite": all(math.isfinite(value) for value in losses),
    "validation_valid": 0.0 <= transformer_val_f1 <= 1.0,
    "baseline_comparison_recorded": math.isfinite(validation_delta),
    "result_is_honest": results["result_type"] == "MEASURED_SMOKE",
    "bilingual_data": {"ar", "en"} <= {row["language"] for row in rows},
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY2_NOTEBOOK3_CORE=PASS")

split_isolation: PASS
baseline_valid: PASS
training_ran: PASS
transformer_weights_updated: PASS
loss_is_finite: PASS
validation_valid: PASS
baseline_comparison_recorded: PASS
result_is_honest: PASS
bilingual_data: PASS
DAY2_NOTEBOOK3_CORE=PASS


## Separate Sentiment Classification Head

Topic and sentiment use separate labels and separate evaluation. This section trains a TF-IDF baseline and a separate Transformer sentiment head, then exports its own Macro-F1 evidence.


In [ ]:
# Separate sentiment label contract.
SENTIMENT_LABELS = sorted({row["sentiment"] for row in rows})
sentiment_label2id = {label: i for i, label in enumerate(SENTIMENT_LABELS)}
sentiment_id2label = {i: label for label, i in sentiment_label2id.items()}
sentiment_train = [row for row in rows if row["split"] == "train"]
sentiment_validation = [row for row in rows if row["split"] == "validation"]
sentiment_test = [row for row in rows if row["split"] == "test"]
assert set(SENTIMENT_LABELS) >= {"negative", "neutral", "positive"}

sentiment_baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LinearSVC(random_state=SEED),
)
sentiment_baseline.fit(
    [row["text"] for row in sentiment_train],
    [row["sentiment"] for row in sentiment_train],
)
baseline_val_pred = sentiment_baseline.predict([row["text"] for row in sentiment_validation]).tolist()
baseline_val_f1_sent = f1_score(
    [row["sentiment"] for row in sentiment_validation],
    baseline_val_pred,
    labels=SENTIMENT_LABELS,
    average="macro",
    zero_division=0,
)
print("Sentiment baseline validation Macro-F1:", round(baseline_val_f1_sent, 4))

sentiment_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(SENTIMENT_LABELS),
    label2id=sentiment_label2id,
    id2label=sentiment_id2label,
)
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in sentiment_model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in sentiment_model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True
sentiment_model.to(DEVICE)

def encode_sentiment(batch):
    encoded = sentiment_tokenizer(
        [row["text"] for row in batch],
        padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    encoded["labels"] = torch.tensor(
        [sentiment_label2id[row["sentiment"]] for row in batch],
        dtype=torch.long, device=DEVICE,
    )
    return encoded

def predict_sentiment(examples):
    sentiment_model.eval()
    predicted = []
    with torch.inference_mode():
        for start in range(0, len(examples), BATCH_SIZE):
            batch = examples[start:start + BATCH_SIZE]
            encoded = sentiment_tokenizer(
                [row["text"] for row in batch],
                padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            indexes = sentiment_model(**encoded).logits.argmax(-1).detach().cpu().tolist()
            predicted.extend(sentiment_id2label[int(index)] for index in indexes)
    return predicted

sentiment_parameters = [p for p in sentiment_model.parameters() if p.requires_grad]
sentiment_optimizer = AdamW(
    sentiment_parameters,
    lr=2e-5 if TRAINING_MODE == "full_finetune" else 1e-4,
)
best_sentiment_f1 = -1.0
best_sentiment_epoch = 0
best_sentiment_state = None
for epoch_index in range(NUM_EPOCHS):
    sentiment_model.train()
    order = np.random.default_rng(SEED + 100 + epoch_index).permutation(len(sentiment_train))
    epoch_losses = []
    for start in range(0, len(order), BATCH_SIZE):
        batch = [sentiment_train[i] for i in order[start:start + BATCH_SIZE]]
        sentiment_optimizer.zero_grad(set_to_none=True)
        loss = sentiment_model(**encode_sentiment(batch)).loss
        assert torch.isfinite(loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sentiment_parameters, 1.0)
        sentiment_optimizer.step()
        epoch_losses.append(float(loss.detach().cpu()))
    val_pred = predict_sentiment(sentiment_validation)
    val_f1 = f1_score(
        [row["sentiment"] for row in sentiment_validation],
        val_pred, labels=SENTIMENT_LABELS, average="macro", zero_division=0,
    )
    if val_f1 > best_sentiment_f1:
        best_sentiment_f1 = float(val_f1)
        best_sentiment_epoch = epoch_index + 1
        best_sentiment_state = {
            name: p.detach().cpu().clone()
            for name, p in sentiment_model.named_parameters() if p.requires_grad
        }
    print(f"sentiment epoch={epoch_index+1:02d} loss={np.mean(epoch_losses):.4f} val_macro_f1={val_f1:.4f}")

if best_sentiment_state:
    with torch.no_grad():
        for name, parameter in sentiment_model.named_parameters():
            if name in best_sentiment_state:
                parameter.copy_(best_sentiment_state[name].to(parameter.device))

sentiment_val_pred = predict_sentiment(sentiment_validation)
sentiment_test_pred = predict_sentiment(sentiment_test)
sentiment_truth = [row["sentiment"] for row in sentiment_test]
sentiment_baseline_test = sentiment_baseline.predict([row["text"] for row in sentiment_test]).tolist()
sentiment_results = {
    "result_type": "MEASURED_SMOKE",
    "task": "sentiment",
    "label_contract": SENTIMENT_LABELS,
    "model_id": MODEL_ID,
    "selected_epoch": best_sentiment_epoch,
    "baseline_validation_macro_f1": float(baseline_val_f1_sent),
    "transformer_validation_macro_f1": float(f1_score(
        [row["sentiment"] for row in sentiment_validation], sentiment_val_pred,
        labels=SENTIMENT_LABELS, average="macro", zero_division=0,
    )),
    "baseline_test_macro_f1": float(f1_score(
        sentiment_truth, sentiment_baseline_test,
        labels=SENTIMENT_LABELS, average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        sentiment_truth, sentiment_test_pred,
        labels=SENTIMENT_LABELS, average="macro", zero_division=0,
    )),
    "test_accuracy": float(accuracy_score(sentiment_truth, sentiment_test_pred)),
}
print(json.dumps(sentiment_results, ensure_ascii=False, indent=2))
(reports_dir / "observed_sentiment.json").write_text(
    json.dumps(sentiment_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("SENTIMENT_HEAD=PASS")
